# QASM3 + DMRG smoke tests

Notebook di test rapido per il nuovo path `QASM3 -> CircuitIR -> DMRG`.

Prerequisiti:
- dipendenze installate da `requirements.txt`
- `qiskit_qasm3_import` disponibile


In [ ]:
# Se serve, decommenta ed esegui:
# !pip install -r ../requirements.txt
# import sys
# print(sys.executable)
# !{sys.executable} -m pip install -r ../requirements.txt


In [ ]:
import os
import sys

sys.path.append("..")

import numpy as np
from src.utils.qasm3_to_ir import qasm3_to_circuit_ir
from src.simulation import DMRG_from_circuit_ir
from src.utils.TN_gen import D_tree, D_mps

In [ ]:
qasm_text = """
OPENQASM 3.0;
include \"stdgates.inc\";
qubit[3] q;
h q[0];
cx q[0], q[1];
x q[1];
cx q[1], q[0];
x q[2];
"""

ir = qasm3_to_circuit_ir(
    qasm_text,
    apply_transpile=True,
    optimization_level=0,
)

print(ir)
for op in ir.operations:
    print(op)

In [ ]:
# Test 1: simulazione TTN su circuito QASM3
network_type = "tree"
network_structure = [1, ir.no_qubits]
compression_steps = 1
no_sweeps = 2

for dmax in [2, 4]:
    D = D_tree(network_structure, dmax)
    np.random.seed(0)
    fidelity, state = DMRG_from_circuit_ir(
        compression_steps=compression_steps,
        no_sweeps=no_sweeps,
        D=D,
        network_structure=network_structure,
        circuit_ir=ir,
        network_type=network_type,
        return_state=True,
    )
    print(f"TTN Dmax={dmax} -> fidelity={fidelity}")
    print(" state =", state)

In [ ]:
# Test 2: simulazione MPS sullo stesso circuito
network_type = "mps"
compression_steps = 1
no_sweeps = 2

for dmax in [2, 4]:
    D = D_mps(ir.no_qubits, dmax)
    np.random.seed(0)
    fidelity = DMRG_from_circuit_ir(
        compression_steps=compression_steps,
        no_sweeps=no_sweeps,
        D=D,
        network_structure=[1, ir.no_qubits],
        circuit_ir=ir,
        network_type=network_type,
    )
    print(f"MPS Dmax={dmax} -> fidelity={fidelity}")

In [ ]:
# Test 3: verifica reject dinamico (atteso errore)
bad_qasm_text = """
OPENQASM 3.0;
include \"stdgates.inc\";
qubit[1] q;
bit[1] c;
measure q[0] -> c[0];
"""

try:
    qasm3_to_circuit_ir(bad_qasm_text, apply_transpile=False)
    print("ERRORE: la conversione doveva fallire")
except Exception as exc:
    print(type(exc).__name__, exc)

In [ ]:
# Test 4: campionamento misure finali (n_samples_final) su stato uniforme a 2 qubit
qasm_sampling_text = """
OPENQASM 3.0;
include "stdgates.inc";
qubit[2] q;
h q[0];
h q[1];
"""

ir_sampling = qasm3_to_circuit_ir(
    qasm_sampling_text,
    apply_transpile=True,
    optimization_level=0,
)

n_samples_final = 8
D = D_tree([1, ir_sampling.no_qubits], 4)

fidelity, state, counts = DMRG_from_circuit_ir(
    compression_steps=1,
    no_sweeps=2,
    D=D,
    network_structure=[1, ir_sampling.no_qubits],
    circuit_ir=ir_sampling,
    network_type="tree",
    return_state=True,
    n_samples_final=n_samples_final,
    seed=123,
    return_counts=True,
)

print("fidelity =", fidelity)
print("state =", state)
print("counts =", counts)

empirical = {b: counts.get(b, 0) / n_samples_final for b in ["00", "01", "10", "11"]}
print("empirical probabilities =", empirical)
print("theory probabilities    =", {"00": 0.25, "01": 0.25, "10": 0.25, "11": 0.25})
